# Notebook 04 — Reliability-Aware Agentic RAG System

## Overview

This notebook demonstrates the reliability extension of the multi-agent RAG baseline developed in Notebook 02. The baseline system retrieves and synthesises answers without evaluating the quality of the underlying evidence, producing hallucinations and overconfident responses in the failure cases identified in the failure analysis.

The extended system introduces a **two-gate reliability architecture** that intercepts the pipeline at two checkpoints: before answer generation (pre-synthesis gate) and after (post-synthesis gate). Between the two gates, an adaptive recovery mechanism modifies retrieval behaviour when evidence quality is insufficient.

## Scope

This notebook covers the following mechanisms:

| Mechanism | Component | Role |
|---|---|---|
| **A** | EvidenceAnalyst | Pre-synthesis gate: estimates whether retrieved evidence is sufficient to proceed |
| **G** | RecoveryAgent | Adaptive recovery: rewrites the query or switches retrieval strategy on failure |
| **E** | AbstentionGate | Unified terminal handler: issues a structured abstention for all failure paths |
| **B** | GroundednessVerifier | Post-synthesis verification: checks whether the answer is grounded in evidence |
| **H** | TrustScorer | Post-synthesis gate: aggregates signals and routes to abstention if trust is insufficient |

> Mechanisms **B** and **H** are integrated as injectable components. Their placeholder interfaces are defined in this notebook; the implementations are provided separately.

## System Architecture

```
Query
  └─► QueryProfiler
           └─► RetrievalCoordinator  (BM25 · Dense · GraphRAG · Weighted RRF)
                    └─► [A] EvidenceAnalyst
                              ├── sufficient ─────────────────────────► AnswerSynthesizer
                              │                                                │
                              │                                          [B] GroundednessVerifier
                              │                                                │
                              │                                          [H] TrustScorer
                              │                                                ├── trust OK ──► Final Response
                              │                                                └── trust low ──┐
                              │                                                                 │
                              ├── insufficient ──► [G] RecoveryAgent                           │
                              │                         ├── retry ──► RetrievalCoordinator     │
                              │                         └── exhausted ──────────────────────┐  │
                              │                                                              │  │
                              └── hard floor ───────────────────────────────────────────────▼──▼
                                                                                     [E] AbstentionGate
```

## Prerequisites

- Notebook 02 completed: baseline retrievers (BM25, Dense, GraphRAG) built and stored in Google Drive
- Shared Drive folder `Adv_GenAI_FS26` accessible and paths configured in Section 2

## Section 1 — Installation

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("ADV_GENAI_RAG_REPO", "https://github.com/dhub100/advanced-genai-rag.git")
REPO_REF = os.environ.get("ADV_GENAI_RAG_REF", "main")
REPO_DIR = pathlib.Path("/content/advanced-genai-rag")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", "sentence-transformers", "faiss-cpu"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", str(REPO_DIR / "baseline" / "package")], check=True)

if "rag" in sys.modules:
    del sys.modules["rag"]
import rag.reliability
print(f"Installed advanced-genai-rag from {REPO_URL} @ {REPO_REF}")
print("rag.reliability module found.")

In [ ]:
import json
import time
import pathlib
import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

## Section 2 — Google Drive & Configuration

Edit the `ROOT` path if your Drive layout differs from the default.

| Variable | Description |
|---|---|
| `PATH_BM25_PICKLE` | Serialised `QEBM25` object built in Notebook 02 |
| `PATH_DENSE_LOADER` | `load_dense_fixed.py` generated during corpus indexing |
| `PATH_GRAG_LOADER` | `load_graphrag.py` generated during graph construction |
| `PATH_QA` | Bilingual benchmark JSON (25 EN + 25 DE question-answer pairs) |
| `PATH_QRELS` | Folder of per-document relevance score JSON files |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ROOT = pathlib.Path("/content/drive/MyDrive/Adv_GenAI_FS26")
REPO_DIR = pathlib.Path("/content/advanced-genai-rag")

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"

PATH_QA_CANDIDATES = [
    REPO_DIR / "baseline/benchmark/benchmark_qa_bilingual.json",
    ROOT / "advanced-genai-rag/baseline/benchmark/benchmark_qa_bilingual.json",
]
PATH_QRELS_CANDIDATES = [
    REPO_DIR / "baseline/benchmark/score/fixed_size",
    ROOT / "advanced-genai-rag/baseline/benchmark/score/fixed_size",
]

PATH_QA = next((p for p in PATH_QA_CANDIDATES if p.exists()), PATH_QA_CANDIDATES[0])
PATH_QRELS = next((p for p in PATH_QRELS_CANDIDATES if p.exists()), PATH_QRELS_CANDIDATES[0])

for name, p in [("BM25", PATH_BM25_PICKLE), ("Dense loader", PATH_DENSE_LOADER),
                ("GraphRAG loader", PATH_GRAG_LOADER), ("QA benchmark", PATH_QA), ("Qrels", PATH_QRELS)]:
    status = "OK" if pathlib.Path(p).exists() else "NOT FOUND"
    print(f"  {status:>9}  {name}: {p}")

## Section 3 — Load Baseline System

Loads the pre-built retrieval components from Google Drive. Refer to Notebook 02 for detailed explanations of each component.

Two compatibility patches are applied before loading:
1. **`torch.is_available` patch** — the generated `load_dense_fixed.py` calls `torch.is_available()` instead of `torch.cuda.is_available()`. We register the alias before the file is imported.
2. **`__main__` pickle patch** — the BM25 object was pickled from a notebook where `BilingualBM25` and `QEBM25` were defined in `__main__`. We re-register the package classes under `__main__` so that `pickle.load` can resolve them.

In [ ]:
import __main__
import torch

# Patch 1: torch.is_available() alias
if not hasattr(torch, 'is_available'):
    torch.is_available = torch.cuda.is_available

# Patch 2: re-register pickled classes in __main__
from rag.retrieval.agents.bm25 import BilingualBM25, QEBM25
from rag.retrieval.translator import EnDeTranslator
__main__.BilingualBM25 = BilingualBM25
__main__.QEBM25 = QEBM25
__main__.EnDeTranslator = EnDeTranslator

from rag.retrieval.agents.bm25 import load_bm25_fixed_qe
from rag.retrieval.agents.dense import DenseRetriever, load_dense_fixed
from rag.retrieval.agents.graph import GraphAgent, load_graph_rag
from rag.retrieval.agents.answer_synthesizer import AnswerSynthesizerAgent
from rag.retrieval.orchestrator import Orchestrator

print("Loading BM25...")
bm25_fixed_qe = load_bm25_fixed_qe(bm25_pickle_path=str(PATH_BM25_PICKLE))
print("Loading Dense retriever...")
dense_fixed   = load_dense_fixed(dense_loader_path=str(PATH_DENSE_LOADER))
print("Loading GraphRAG...")
graph_rag     = load_graph_rag(loader_path=str(PATH_GRAG_LOADER))
print("Loading AnswerSynthesizer (Mistral-7B)...")
synthesizer   = AnswerSynthesizerAgent()

baseline_orchestrator = Orchestrator(
    bm25=bm25_fixed_qe, dense=dense_fixed,
    graph=graph_rag, synthesizer=synthesizer
)
print("✓ Baseline system ready.")

## Section 4 — Evidence Sufficiency Estimation [Mechanism A]

### Problem

The baseline passes retrieved documents directly to Mistral-7B without assessing whether they contain sufficient relevant information. This is the root cause of two failure types identified in the failure analysis: hallucinations arise when weak evidence is treated as valid input, and overconfidence failures arise when the system returns a confident answer based on tangentially related material.

### Solution: EvidenceSufficiencyChecker

Mechanism A is implemented through the `EvidenceSufficiencyChecker`. Its purpose is to determine whether retrieved context is sufficient **before** answer generation. The current implementation keeps the original scoring layer and adds a second gating / routing layer so that topically related chunks are not mistaken for answer-bearing evidence.

The scoring layer evaluates four independent signals on the retrieved document set:

| Signal | Method | Threshold |
|---|---|---|
| `semantic_coverage` | Average cosine similarity between the query embedding and top-k document embeddings (E5-large) | ≥ 0.50 |
| `chunk_support_count` | Number of documents whose individual cosine similarity to the query exceeds 0.40 | ≥ 3 |
| `aspect_coverage_ratio` | Fraction of query key-terms (stopword-filtered) present in the combined retrieved text | ≥ 0.50 |
| `temporal_valid` | If the query contains a year: at least one retrieved document must reference it; years outside 1850–2030 fail immediately | boolean |
| `sufficiency_score` | Weighted composite of semantic, chunk, and aspect signals | ≥ 0.45 |

The gating layer then maps the score and diagnostics to a routing decision:

- `answer`: evidence is sufficient and explicitly answer-bearing
- `recover`: evidence is related but still incomplete or weak
- `clarify`: the query is ambiguous or underspecified
- `abstain`: evidence is too weak for reliable recovery

Additional rules now catch missing critical aspects, ambiguous wording, temporal answer-support failures, and answer-type support failures. A **hard floor** of 0.20 still bypasses recovery and triggers immediate abstention.

### Semantic Coverage Threshold

The `semantic_coverage` threshold is set at **0.50** based on the empirical interpretation of cosine similarity values for `multilingual-E5-large` in a retrieval context:

| Similarity Range | Interpretation |
|---|---|
| 0.70 – 1.00 | Near-paraphrase — retrieved text directly addresses the query |
| 0.50 – 0.70 | Same topic — document likely contains the relevant facts |
| 0.35 – 0.50 | Tangentially related — same domain but different subtopic |
| 0.20 – 0.35 | Weakly related — surface-level keyword overlap only |

A threshold of 0.35 was found to be too permissive during testing: topically adjacent documents pass the check even when they cannot answer the question. The threshold of 0.50 ensures that the average retrieved document is genuinely on-topic. A value above 0.55 was avoided as it risks false abstentions on cross-lingual queries where vocabulary mismatch slightly depresses similarity scores.

> The `EvidenceSufficiencyChecker` reuses the `multilingual-E5-large` embeddings already loaded by the Dense retriever. No additional model is loaded.

In [ ]:
import numpy as np
from rag.reliability.evidence_sufficiency import EvidenceSufficiencyChecker, EvidenceReport

# DenseRetriever wraps a ChromaDB store whose embedding function is accessible
# via store._embedding_function (a HuggingFaceEmbeddings instance).
def embed_fn(texts: list[str]) -> np.ndarray:
    return np.array(dense_fixed.store._embedding_function.embed_documents(texts))

checker = EvidenceSufficiencyChecker(embed_fn=embed_fn)
print("✓ EvidenceSufficiencyChecker ready.")

In [ ]:
def show_evidence_report(query: str, strategy: str = "confidence", top_k: int = 5):
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=top_k)
    report = checker.check(query, result["documents"])

    print(f"Query  : {query}")
    print(f"{'─'*62}")
    print(f"  semantic_coverage     : {report.semantic_coverage:.3f}   (threshold ≥ 0.50)")
    print(f"  chunk_support_count   : {report.chunk_support_count}       (threshold ≥ 3)")
    print(f"  aspect_coverage_ratio : {report.aspect_coverage_ratio:.3f}   (threshold ≥ 0.50)")
    print(f"  temporal_valid        : {report.temporal_valid}")
    print(f"{'─'*62}")
    print(f"  sufficiency_score     : {report.sufficiency_score:.3f}   → sufficient = {report.sufficient}")
    print(f"  failure_type          : {report.failure_type}")
    if report.missing_aspects:
        print(f"  missing_aspects       : {report.missing_aspects}")
    print(f"  routing_decision      : {report.routing_decision}")
    print(f"  decision_reason       : {report.decision_reason}")
    return report

In [ ]:
print("=" * 62)
print("EXAMPLE 1 — likely sufficient evidence")
print("=" * 62)
_ = show_evidence_report("Who at ETH received ERC grants?")

print()
print("=" * 62)
print("EXAMPLE 2 — known baseline failure (hallucination case)")
print("=" * 62)
_ = show_evidence_report("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("EXAMPLE 3 — ambiguous query")
print("=" * 62)
_ = show_evidence_report("What research is ETH famous for?")

## Section 5 — Recovery Agent [Mechanism G — Adaptive Requirement]

### Design rationale

A generic retry with an unchanged query would explore no new evidence. The `RecoveryAgent` diagnoses which signal was the dominant failure contributor and selects a targeted corrective action that modifies the system's subsequent retrieval behaviour. This satisfies the adaptation requirement: the orchestration logic changes dynamically based on observed evidence quality rather than following a fixed script.

The recovery policy maps each failure type to a distinct corrective action:

| `failure_type` | Recovery action | Rationale |
|---|---|---|
| `coverage_low` | LLM-based query rewrite | Low aspect coverage indicates the query is too narrow; rephrasing with synonyms or broader terms improves recall |
| `few_docs` | Strategy switch (Confidence → Voting → Waterfall) | A different fusion strategy surfaces complementary results from underutilised agents |
| `low_similarity` | PRF expansion via QEBM25 | Low semantic similarity indicates a vocabulary mismatch; pseudo-relevance feedback expands the query using top-document terms |
| `exhausted` | Signal AbstentionGate | Further attempts are unlikely to yield better evidence given the retrieval corpus |

A maximum of **two recovery attempts** is enforced to bound worst-case latency.

### Query rewriting

LLM-based query rewriting uses the OpenAI API and requires a key stored as a Colab Secret:
1. Click the **🔑 Secrets** icon in the left Colab sidebar
2. Add a secret named `OPENAI_API_KEY` with your key as the value
3. Toggle **Notebook access** on

The key is stored in your personal Colab account and is never written to the notebook file. Without a key, a rule-based fallback appends the missing key-terms to the original query.

In [ ]:
from rag.reliability.recovery import RecoveryAgent

openai_client = None
try:
    from google.colab import userdata
    openai_api_key = userdata.get("OPENAI_API_KEY")
    if openai_api_key:
        from openai import OpenAI
        openai_client = OpenAI(api_key=openai_api_key)
        print("✓ OpenAI client ready — LLM-based query rewriting enabled.")
    else:
        print("ℹ  OPENAI_API_KEY secret not set — rule-based query rewrite fallback will be used.")
except Exception:
    print("ℹ  Could not read Colab secrets — rule-based query rewrite fallback will be used.")

recovery_agent = RecoveryAgent(max_retries=2, openai_client=openai_client)
print("✓ RecoveryAgent ready.")

In [ ]:
# These queries are taken directly from the baseline failure analysis.
# All four fail consistently across every retrieval strategy in the baseline,
# making them reliable triggers for recovery behaviour.

RECOVERY_CASES = [
    {
        "query":    "who was president of ETH in 2003?",
        "gold":     "olaf kübler",
        "expected": "temporal_answer_not_supported — topic match without local year-to-role support; "
                    "query rewrite expected on attempt 1",
    },
    {
        "query":    "who were the rectors of ETH between 2017 and 2022?",
        "gold":     "sarah springman, günther dissertori",
        "expected": "missing_query_aspect — aspect terms absent from retrieved text; "
                    "query rewrite expected on attempt 1",
    },
    {
        "query":    "what is e-Sling?",
        "gold":     "4-seat electric airplane built by ETH students",
        "expected": "low_semantic_coverage — document exists in corpus in German only; "
                    "PRF expansion expected to improve cross-lingual match",
    },
    {
        "query":    "who was president of ETH in 2045?",
        "gold":     "N/A — future date, no document can exist in corpus",
        "expected": "temporal_mismatch — year 2045 is outside corpus range (1850–2030); "
                    "hard evidence mismatch prevents direct answering",
    },
]

def show_recovery_actions(query: str, strategy: str = "confidence"):
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=5)
    report = checker.check(query, result["documents"])

    print(f"Query        : {query}")
    print(f"sufficient   : {report.sufficient}  |  score={report.score:.3f}  |  failure='{report.failure_type}'")
    print(f"route        : {report.routing_decision}")
    print(f"reason       : {report.decision_reason}")
    print(f"temporal_valid: {report.temporal_valid}")

    if report.routing_decision in {"answer", "clarify", "abstain"}:
        print(f"  → Routing decision is '{report.routing_decision}'.")
        return

    current_strategy = strategy
    current_query    = query
    for attempt in range(recovery_agent.max_retries + 1):
        action = recovery_agent.choose_action(report, attempt=attempt, current_strategy=current_strategy)
        print(f"  {action.trace_entry}")
        if action.type == "abstain":
            break
        if action.type == "rewrite_query":
            current_query = recovery_agent.rewrite_query(current_query, report.missing_aspects)
            print(f"    → Rewritten query: '{current_query}'")
        elif action.type == "switch_strategy":
            current_strategy = action.new_strategy


for case in RECOVERY_CASES:
    print("=" * 66)
    print(f"Expected : {case['expected']}")
    print(f"Gold     : {case['gold']}")
    print("=" * 66)
    show_recovery_actions(case["query"])
    print()

### Observed Limitation of Mechanism A — and the Fix

**Observation.** When testing the query *"who was president of ETH in 2045?"*, the EvidenceAnalyst returned:

```
sufficient : True  |  score=0.917  |  failure='ok'
→ Evidence is sufficient — proceeds to synthesis.
```

A score of 0.917 for a question about a future date that no corpus document can answer revealed a structural limitation of the pre-synthesis evidence check:

> **A measures semantic and lexical coverage — not temporal or factual validity.**

The corpus contains documents about ETH presidents and leadership, so retrieved chunks are topically related to the query. The three original signals (semantic coverage, chunk support, aspect coverage) all pass because they measure *topic proximity*, not *whether the specific fact exists*. A had no way to distinguish between *"relevant documents retrieved"* and *"the requested fact exists in those documents"*.

---

**Root cause.** The composite score formula does not account for the temporal scope of the query. A future year is indistinguishable from any other keyword — it does not lower semantic similarity, it does not reduce aspect coverage, and it does not reduce chunk count.

---

**Fix: `temporal_valid` signal added to Mechanism A.**

A fourth signal was added to `EvidenceSufficiencyChecker`:

- Extract all 4-digit years from the query using a regex pattern
- If a year is found, check whether at least one retrieved document references that year
- Years outside the plausible corpus range (1850–2030) fail immediately
- When `temporal_valid = False`, the composite score is overridden to insufficient and `failure_type` is set to `temporal_mismatch`

After the fix, the same query now returns:

```
sufficient : False  |  score=0.44  |  failure='temporal_mismatch'
→ Recovery: query rewrite scheduled (missing: 2045)
```

---

**Remaining boundary.** Temporal validity closes the gap for date-scoped queries. For queries where the year is plausible but the specific fact is simply absent from the corpus, B (Groundedness) remains the necessary complement — A passes the query to synthesis, and B detects that the generated answer is unsupported by the retrieved text.

## Section 6 — Abstention Gate [Mechanism E]

### Design rationale

Returning an empty string or a generic "I don't know" provides no actionable information to the user or to the evaluation pipeline. The `AbstentionGate` produces a structured `AbstentionResponse` containing a machine-readable `trigger`, a human-readable `reason`, the list of `missing_aspects` that were absent from the evidence, a `confidence` of 0.0, and the full decision `trace` accumulated up to that point.

This structure enables fine-grained evaluation metrics — in particular the distinction between **correct abstentions** (the answer was genuinely unavailable) and **false abstentions** (the answer existed but thresholds were miscalibrated).

### Two entry paths

The AbstentionGate is the **unified terminal failure state**, reached from two independent paths:

| Path | Trigger | Method |
|---|---|---|
| Retrieval failure | A hard floor or G exhausted | `abstain_evidence(query, report, trace)` |
| Synthesis failure | H trust score below threshold | `abstain_low_trust(query, trust_score, groundedness_score, trace)` |

## Section 7 — Groundedness Verification [Mechanism B]

> **This section is a placeholder for Robin Girardin (Mechanism B).**

### Role in the pipeline

The GroundednessVerifier is invoked **after** answer synthesis, once the EvidenceAnalyst has already confirmed that the retrieved evidence is sufficient. Its role is to check whether the claims in the generated answer are actually supported by the retrieved documents — catching cases where Mistral-7B has hallucinated or over-generalised despite receiving adequate context.

### Expected interface

The `ReliableOrchestrator` calls B via:

```python
groundedness_score = grounder.check(query, answer, docs)
```

| Parameter | Type | Description |
|---|---|---|
| `query` | `str` | The original user query |
| `answer` | `str` | The answer generated by `AnswerSynthesizerAgent` |
| `docs` | `list[Document]` | The retrieved documents passed to the synthesiser |
| **Returns** | `float ∈ [0, 1]` | Groundedness score: 1.0 = fully grounded, 0.0 = no support |

### Integration point

Once implemented, replace `groundedness_verifier = None` in Section 8 with an instance of your class. The orchestrator will call `.check()` automatically and pass the score to H (TrustScorer).

### Suggested approach

- Claim-level or sentence-level alignment: split the answer into claims, check each against retrieved text
- Answer-to-source overlap using NLI or embedding similarity
- Verifier-style prompting: ask an LLM whether each claim is supported by the context

In [ ]:
# ══════════════════════════════════════════════════════════════
# PLACEHOLDER — Mechanism B (GroundednessVerifier)
# Implement your class here and assign it to groundedness_verifier
# ══════════════════════════════════════════════════════════════

# from rag.reliability.groundedness import GroundednessVerifier
# groundedness_verifier = GroundednessVerifier(...)

# Minimal stub — replace with real implementation
class GroundednessVerifierStub:
    """Stub — returns None so the orchestrator bypasses the post-synthesis gate."""
    def check(self, query: str, answer: str, docs: list) -> float:
        raise NotImplementedError("Mechanism B not yet implemented.")

# Leave as None until B is ready — the orchestrator skips the gate gracefully
groundedness_verifier = None
print("ℹ  groundedness_verifier = None  (Mechanism B placeholder)")

## Section 8 — Trust Scoring [Mechanism H]

> **This section is a placeholder for Daniel Huber (Mechanism H).**

### Role in the pipeline

The TrustScorer is the **post-synthesis quality gate**. It aggregates the `sufficiency_score` produced by A and the `groundedness_score` produced by B into a single `trust_score`, then decides whether to return the answer or route to the AbstentionGate. This gives H a dual role: it is both a scorer (combining two independent signals) and a decision gate (the final checkpoint before any answer leaves the system).

### Expected interface

The `ReliableOrchestrator` calls H via:

```python
trust_score = trust_scorer.score(evidence_report, groundedness_score)
```

| Parameter | Type | Description |
|---|---|---|
| `evidence_report` | `EvidenceReport` | Output of the EvidenceAnalyst (A); contains `sufficiency_score` and all raw signals |
| `groundedness_score` | `float \| None` | Output of the GroundednessVerifier (B); `None` if B is not yet available |
| **Returns** | `float ∈ [0, 1]` | Trust score: answers with `trust_score < 0.45` are routed to abstention |

### Key fields available from EvidenceReport

```python
evidence_report.score                # composite sufficiency score (0–1)
evidence_report.semantic_coverage    # avg cosine similarity
evidence_report.chunk_support_count  # number of supporting chunks
evidence_report.aspect_coverage_ratio
evidence_report.temporal_valid       # False if year in query not found in docs
```

### Integration point

Once implemented, replace `trust_scorer = None` in Section 9 with an instance of your class. The orchestrator calls `.score()` after B and routes to E if the result is below 0.45.

### Suggested approach

- Weighted average of `sufficiency_score` (from A) and `groundedness_score` (from B)
- Calibrate weights empirically on the benchmark: heavier weight on groundedness if B is reliable
- Consider penalising answers where `temporal_valid = False` even if they pass the composite score

In [ ]:
# ══════════════════════════════════════════════════════════════
# PLACEHOLDER — Mechanism H (TrustScorer)
# Implement your class here and assign it to trust_scorer
# ══════════════════════════════════════════════════════════════

# from rag.reliability.trust_scorer import TrustScorer
# trust_scorer = TrustScorer(...)

# Minimal stub — replace with real implementation
class TrustScorerStub:
    """Stub — raises NotImplementedError when called."""
    def score(self, evidence_report, groundedness_score) -> float:
        raise NotImplementedError("Mechanism H not yet implemented.")

# Leave as None until H is ready — the orchestrator skips the post-synthesis gate gracefully
trust_scorer = None
print("ℹ  trust_scorer = None  (Mechanism H placeholder)")

In [ ]:
from rag.reliability.abstention import AbstentionMechanism, AbstentionResponse

abstainer = AbstentionMechanism()
print("✓ AbstentionMechanism ready.")

In [ ]:
def show_abstention_evidence(query: str):
    result = baseline_orchestrator.run(strategy="confidence", query=query, top_k=5)
    report = checker.check(query, result["documents"])
    trace  = [
        f"EvidenceCheck: score={report.score:.3f}, failure='{report.failure_type}'",
        "Recovery attempts exhausted."
    ]
    abstention = abstainer.abstain_evidence(query, report, trace)

    print(f"Query      : {query}")
    print(f"Trigger    : {abstention.trigger}")
    print(f"Reason     : {abstention.reason}")
    print(f"Missing    : {abstention.missing_aspects}")
    print(f"Confidence : {abstention.confidence}")
    print("Trace:")
    for e in abstention.trace:
        print(f"  {e}")


print("=" * 62)
print("ABSTENTION — evidence_failure path (A → G → E)")
print("=" * 62)
show_abstention_evidence("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("ABSTENTION — trust_failure path (B → H → E)")
print("=" * 62)
# ─────────────────────────────────────────────────────────────
# Introduce code here for H (TrustScorer)
# Replace the placeholder values with the real trust_score and
# groundedness_score produced by H and B respectively.
# ─────────────────────────────────────────────────────────────
abstention_trust = abstainer.abstain_low_trust(
    query="Who was president of ETH in 2003?",
    trust_score=0.0,          # placeholder — replace with H output
    groundedness_score=0.0,   # placeholder — replace with B output
    trace=["[H placeholder] trust_score below threshold"]
)
print(f"Trigger    : {abstention_trust.trigger}")
print(f"Reason     : {abstention_trust.reason}")
print(f"Trust      : {abstention_trust.trust_score}")

## Section 9 — Reliable Orchestrator (A + G + E)

The `ReliableOrchestrator` is the policy controller that coordinates all reliability mechanisms. It wraps the existing `Orchestrator` and inserts the evidence-based loop between retrieval and synthesis.

`groundedness_verifier` and `trust_scorer` are passed in from Sections 7 and 8. When both are `None`, the post-synthesis gate (B → H → E) is bypassed and A, G, and E operate independently — allowing the pre-synthesis pipeline to be evaluated before B and H are complete.

### Decision policy summary

| State | Condition | Next action |
|---|---|---|
| After retrieval | `routing_decision = answer` | Proceed to synthesis |
| After retrieval | `routing_decision = recover` and attempts < 2 | Recover → retry retrieval |
| After retrieval | `routing_decision = clarify` | Surface clarification need and stop before synthesis |
| After retrieval | `routing_decision = abstain` or hard floor hit | Abstain immediately |
| After retrieval | `routing_decision = recover` and attempts ≥ 2 | Abstain |
| After synthesis | `trust_score ≥ 0.45` | Return final response |
| After synthesis | `trust_score < 0.45` | Abstain (trust_failure) |

In [ ]:
from rag.reliability.reliable_orchestrator import ReliableOrchestrator, ReliableResponse

reliable_orchestrator = ReliableOrchestrator(
    orchestrator=baseline_orchestrator,
    embed_fn=embed_fn,
    max_retries=2,
    openai_client=openai_client,
    groundedness_verifier=groundedness_verifier,   # set in Section 7
    trust_scorer=trust_scorer,                     # set in Section 8
)
print("✓ ReliableOrchestrator ready.")

In [ ]:
def run_and_show(query: str, strategy: str = "confidence"):
    t0       = time.time()
    response = reliable_orchestrator.run(query=query, strategy=strategy, top_k=5)
    elapsed  = time.time() - t0

    print(f"Query      : {query}")
    print(f"Abstained  : {response.abstained}  |  Recoveries: {response.recovery_attempts}  |  {elapsed:.2f}s")

    if response.abstained:
        print(f"Trigger    : {response.abstention.trigger}")
        print(f"Reason     : {response.abstention.reason}")
    else:
        print(f"Answer     : {response.answer}")
        if response.evidence_report:
            print(f"Evidence   : score={response.evidence_report.score:.3f}")

    print("Trace (last 5 entries):")
    for entry in response.trace[-5:]:
        print(f"  {entry}")
    print()

In [ ]:
# Four representative cases drawn from the baseline failure analysis.
# The baseline returns hallucinated or "NOT FOUND" answers for all failure cases.
# Expected behaviour of the reliable system is stated for each.

TEST_CASES = [
    {
        "query":    "who at ETH received ERC grants?",
        "gold":     "tilman esslinger, ursula keller, and others",
        "note":     "Expected: recover if critical aspect 'grants' is missing from retrieved evidence",
    },
    {
        "query":    "who was president of ETH in 2003?",
        "gold":     "olaf kübler",
        "note":     "Baseline: hallucination ('Günther Huber'). "
                    "Expected: recover because year-to-role support is not explicit",
    },
    {
        "query":    "who were the rectors of ETH between 2017 and 2022?",
        "gold":     "sarah springman, günther dissertori",
        "note":     "Baseline: all strategies return NOT FOUND. "
                    "Expected: query rewrite recovery or abstention",
    },
    {
        "query":    "what is e-Sling?",
        "gold":     "4-seat electric airplane built by 20 ETH students",
        "note":     "Baseline: all strategies return NOT FOUND (doc is German-only). "
                    "Expected: PRF expansion recovery or abstention",
    },
    {
        "query":    "how do birds learn new songs?",
        "gold":     "incremental syllable adaptation, similar to child language acquisition",
        "note":     "Corpus gap — relevant document not in subsample. "
                    "Expected: abstention after recovery attempts exhausted",
    },
]

for case in TEST_CASES:
    print("=" * 66)
    print(f"Note: {case['note']}")
    print("=" * 66)
    run_and_show(case["query"])
    print(f"Gold: {case['gold']}")
    print()

## Section 8 — Benchmark Evaluation

Runs the reliable orchestrator over all 50 benchmark queries (25 EN + 25 DE) and reports abstention rate, recovery statistics, and per-query outcomes.

In [ ]:
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions, gold_answers = [], []
for x in qa_data:
    questions.append(x["question"])
    gold_answers.append(x["answer"])
    questions.append(x["question_de"])
    gold_answers.append(x["answer_de"])

print(f"Loaded {len(questions)} queries ({len(qa_data)} bilingual pairs).")

In [ ]:
reliable_results = []

for q, a in tqdm(zip(questions, gold_answers), total=len(questions), desc="Reliable pipeline"):
    t0 = time.time()
    r  = reliable_orchestrator.run(query=q, strategy="confidence", top_k=5)
    reliable_results.append({
        "query":      q,
        "gold":       a,
        "answer":     r.answer,
        "abstained":  r.abstained,
        "trigger":    r.abstention.trigger if r.abstained else None,
        "recoveries": r.recovery_attempts,
        "ev_score":   r.evidence_report.score if r.evidence_report else None,
        "latency":    time.time() - t0,
    })

rel_df = pd.DataFrame(reliable_results)
print("Done.")

In [ ]:
n            = len(reliable_results)
abstained_df = rel_df[rel_df["abstained"]]
recovered_df = rel_df[rel_df["recoveries"] > 0]
answered_df  = rel_df[~rel_df["abstained"]]

print(f"{'─'*40}")
print(f"  Total queries    : {n}")
print(f"  Answered         : {len(answered_df):3d}  ({len(answered_df)/n:.1%})")
print(f"  Abstained        : {len(abstained_df):3d}  ({len(abstained_df)/n:.1%})")
print(f"  Had recovery     : {len(recovered_df):3d}  ({len(recovered_df)/n:.1%})")
print(f"  Avg latency      : {rel_df['latency'].mean():.2f}s")
print(f"  Avg ev_score     : {rel_df['ev_score'].dropna().mean():.3f}")
print(f"{'─'*40}")

if len(abstained_df) > 0:
    print("\nAbstention trigger breakdown:")
    print(abstained_df["trigger"].value_counts().to_string())
    print("\nAbstained queries:")
    for _, row in abstained_df.iterrows():
        ev = f"{row['ev_score']:.2f}" if row['ev_score'] is not None else "N/A"
        print(f"  [{row['trigger']:20s}] ev={ev}  {row['query'][:60]}")

In [ ]:
if len(recovered_df) > 0:
    answered_after  = recovered_df[~recovered_df["abstained"]]
    abstained_after = recovered_df[recovered_df["abstained"]]

    print("Recovery outcomes:")
    print(f"  Answered after recovery  : {len(answered_after)}")
    print(f"  Abstained after recovery : {len(abstained_after)}")

    if len(answered_after) > 0:
        print("\nQueries where recovery produced an answer:")
        for _, row in answered_after.iterrows():
            print(f"  [recoveries={int(row['recoveries'])}] ev={row['ev_score']:.2f}  {row['query'][:60]}")
else:
    print("No recovery attempts triggered on this benchmark run.")